# Load DIV30 and DIV90 Raw 10x Samples to AnnData

This notebook defines a small in-notebook builder class for raw 10x sample loading. The immediate focus is DIV30, but the same object works for DIV90 or both time points by changing `target_divs`.

The builder reads per-sample 10x matrices from `metadata/div30_div90_sample_id_to_biolabel_map.tsv`, adds sample metadata to `.obs`, makes cell IDs unique by prefixing the 10x barcode with `run_sample_id`, and can return either a standard combined `AnnData` or, if `snapatac2` is available, a backed `AnnDataSet`.


In [1]:
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Sequence

import anndata as ad
import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc

In [2]:
@dataclass
class Raw10xAnnDataBuilder:
    """Minimal teaching builder: only methods used in this tutorial."""

    project_root: Path
    target_divs: Sequence[str] = ("DIV30",)
    target_run_sample_ids: Optional[Sequence[str]] = None
    sample_map_name: str = "metadata/div30_div90_sample_id_to_biolabel_map.tsv"
    strict_missing_matrix_dirs: bool = True

    @property
    def sample_map_tsv(self) -> Path:
        """Compute sample map path only (no file read)."""
        return self.project_root / self.sample_map_name

    def sample_table(self) -> pd.DataFrame:
        """Read/filter sample metadata, map run_sample_id -> cell_line, and derive matrix paths."""
        sample_metadata_df = pd.read_csv(self.sample_map_tsv, sep="\t")
        required = {"DIV", "run_sample_id", "biological_label", "per_sample_metrics_csv"}
        missing = required.difference(sample_metadata_df.columns)
        if missing:
            raise ValueError(f"Sample map is missing required columns: {sorted(missing)}")

        sample_metadata_df = sample_metadata_df[sample_metadata_df["DIV"].isin(self.target_divs)].copy()

        if self.target_run_sample_ids is not None:
            sample_metadata_df = sample_metadata_df[
                sample_metadata_df["run_sample_id"].isin(self.target_run_sample_ids)
            ].copy()
            found = set(sample_metadata_df["run_sample_id"].astype(str))
            missing_ids = [sid for sid in self.target_run_sample_ids if sid not in found]
            if missing_ids:
                raise ValueError(f"Missing requested run_sample_id values in sample map: {missing_ids}")

        if sample_metadata_df.empty:
            raise ValueError("No samples matched target_divs/target_run_sample_ids.")

        # Normalize replicate-level labels (e.g., H9_rep1) to canonical cell lines (H9/79B/2E).
        sample_metadata_df["cell_line"] = sample_metadata_df["biological_label"].astype("string").str.extract(
            r"^(H9|79B|2E)",
            expand=False,
        )

        invalid_rows = sample_metadata_df[sample_metadata_df["cell_line"].isna()][["run_sample_id", "biological_label"]]
        if not invalid_rows.empty:
            examples = invalid_rows.head(10).to_dict(orient="records")
            raise ValueError(
                "Could not map some biological_label values to expected cell_line prefixes "
                "(H9, 79B, 2E). Examples: "
                f"{examples}"
            )

        sample_metadata_df["DIV"] = pd.Categorical(sample_metadata_df["DIV"], categories=self.target_divs, ordered=True)
        if self.target_run_sample_ids is not None:
            sample_metadata_df["run_sample_id"] = pd.Categorical(
                sample_metadata_df["run_sample_id"],
                categories=self.target_run_sample_ids,
                ordered=True,
            )

        sample_metadata_df = sample_metadata_df.sort_values(["DIV", "run_sample_id"]).reset_index(drop=True)
        sample_metadata_df["matrix_dir"] = sample_metadata_df["per_sample_metrics_csv"].map(
            lambda p: str(Path(p).parent / "count" / "sample_filtered_feature_bc_matrix")
        )

        return sample_metadata_df[["DIV", "run_sample_id", "cell_line", "matrix_dir"]]

    def _load_component_adatas(self):
        """Load per-sample AnnData objects and return a dict keyed by run_sample_id."""
        sample_metadata_df = self.sample_table()
        sample_adatas_by_run = {}
        missing_dirs = []

        for _, row in sample_metadata_df.iterrows():
            matrix_dir = Path(row["matrix_dir"])
            if not matrix_dir.exists():
                missing_dirs.append(str(matrix_dir))
                continue

            one_sample_adata = sc.read_10x_mtx(matrix_dir, var_names="gene_symbols", make_unique=True)
            run_sample_id = str(row["run_sample_id"])

            one_sample_adata.obs_names = [f"{run_sample_id}:{barcode}" for barcode in one_sample_adata.obs_names]
            one_sample_adata.obs["DIV"] = str(row["DIV"])
            one_sample_adata.obs["run_sample_id"] = run_sample_id
            one_sample_adata.obs["cell_line"] = str(row["cell_line"])
            one_sample_adata.obs["matrix_dir"] = str(matrix_dir)
            sample_adatas_by_run[run_sample_id] = one_sample_adata

        if missing_dirs:
            message = "Missing per-sample 10x matrix directories:\n" + "\n".join(f" - {p}" for p in missing_dirs)
            if self.strict_missing_matrix_dirs:
                raise FileNotFoundError(message)
            print(message)

        if not sample_adatas_by_run:
            raise FileNotFoundError("No per-sample 10x matrix directories were found.")

        return sample_metadata_df, sample_adatas_by_run

    def _finalize_combined(self, combined_adata: ad.AnnData, sample_metadata_df: pd.DataFrame) -> ad.AnnData:
        """Apply common post-concatenation typing, checks, and provenance fields."""
        if not combined_adata.obs_names.is_unique:
            duplicated = combined_adata.obs_names[combined_adata.obs_names.duplicated()].unique()[:10].tolist()
            raise ValueError(f"Combined AnnData has duplicated obs_names. Examples: {duplicated}")

        combined_adata.obs["DIV"] = pd.Categorical(combined_adata.obs["DIV"], categories=self.target_divs, ordered=True)
        combined_adata.obs["run_sample_id"] = pd.Categorical(
            combined_adata.obs["run_sample_id"],
            categories=sample_metadata_df["run_sample_id"].astype(str).tolist(),
            ordered=True,
        )
        combined_adata.obs["cell_line"] = combined_adata.obs["cell_line"].astype("string")
        combined_adata.obs["matrix_dir"] = combined_adata.obs["matrix_dir"].astype("string")

        combined_adata.uns["sample_map_tsv"] = str(self.sample_map_tsv)
        combined_adata.uns["target_divs"] = list(self.target_divs)
        combined_adata.uns["combine_logic"] = (
            "Per-sample 10x matrices were read separately, annotated, barcode-prefixed by "
            "run_sample_id, and concatenated along observations with an outer gene join."
        )
        return combined_adata

    def combined_anndata(self, join: str = "outer") -> ad.AnnData:
        """Original combine path: concatenate loaded components without creating a batch column."""
        sample_metadata_df, sample_adatas_by_run = self._load_component_adatas()
        combined_adata = ad.concat(
            list(sample_adatas_by_run.values()),
            axis=0,
            join=join,
            merge="same",
            fill_value=0,
            index_unique=None,
        )
        return self._finalize_combined(combined_adata, sample_metadata_df)

    def combined_anndata_concat(self, join: str = "outer", label: str = "batch") -> ad.AnnData:
        """Alternative combine path using anndata.concat with a batch key in .obs."""
        sample_metadata_df, sample_adatas_by_run = self._load_component_adatas()
        combined_adata = ad.concat(
            sample_adatas_by_run,
            axis=0,
            join=join,
            merge="same",
            fill_value=0,
            index_unique=None,
            label=label,
        )

        # Ensure batch values are run_sample_id values, and keep run_sample_id explicit.
        if label in combined_adata.obs.columns:
            combined_adata.obs[label] = combined_adata.obs[label].astype("string")
            combined_adata.obs["run_sample_id"] = combined_adata.obs[label].astype("string")

        return self._finalize_combined(combined_adata, sample_metadata_df)

    def save_per_sample_h5ad(self, output_dir: Optional[Path] = None, compression: str = "gzip") -> pd.DataFrame:
        """Save one AnnData file per run_sample_id and return a write manifest."""
        sample_metadata_df, sample_adatas_by_run = self._load_component_adatas()

        if output_dir is None:
            output_dir = self.project_root / "results" / "python_anndata" / "per_sample_h5ad"
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

        records = []
        sample_meta_by_run = sample_metadata_df.set_index("run_sample_id")

        for run_sample_id, one_sample_adata in sample_adatas_by_run.items():
            out_path = output_dir / f"{run_sample_id}.h5ad"
            one_sample_adata.write_h5ad(out_path, compression=compression)
            records.append(
                {
                    "run_sample_id": run_sample_id,
                    "cell_line": str(sample_meta_by_run.loc[run_sample_id, "cell_line"]),
                    "n_cells": int(one_sample_adata.n_obs),
                    "n_genes": int(one_sample_adata.n_vars),
                    "h5ad_path": str(out_path),
                }
            )

        write_manifest_df = pd.DataFrame(records).sort_values("run_sample_id").reset_index(drop=True)
        return write_manifest_df

## Builder Instantiation Tutorial (Minimal Method Version)

This notebook now uses a simplified class that keeps only the methods needed for your learning path:

1. `sample_map_tsv` (property)
2. `sample_table()`
3. `combined_anndata()`
4. `combined_anndata_concat()`
5. `save_per_sample_h5ad()`

Methods like `read_one_sample`, `component_anndatas`, `cell_count_tables`, and `anndata_set` were intentionally removed for clarity. Their logic is now either unnecessary for this tutorial or inlined.

### Seurat vs Scanpy data storage model

Here comes another difference from Seurat. The R package stores raw data, scaled data and variable genes information in separate slots, Scanpy instead keeps only one snapshot of the data.

In practice with AnnData, this means:

- active matrix is typically in `.X`
- optional alternatives can be stored in `.layers[...]`
- raw snapshot can be stored in `.raw`

### Naming clarification for cell identity and runtime variables

In this notebook, we now use `cell_line` as the explicit column name.

- Source metadata column: `biological_label`
- Tutorial/output column: `cell_line`
- Mapping rule: each `run_sample_id` maps directly to one canonical `cell_line` value.

Variable naming is now explicit:

- `anndata_builder`: configuration/helper object (not the expression matrix)
- `sample_metadata_df`: sample-level metadata table
- `combined_adata`: combined AnnData object in memory

### Where per-sample AnnData files are saved

Standalone raw save utility default location:

`$PROJECT_ROOT/raw_adata`

You should export `PROJECT_ROOT` first, for example:

`export PROJECT_ROOT=/nfs/turbo/umms-parent/mgeo_neuron_scrnaseq_projectfolder`

Safety behavior:

- default mode is `dry_run=True` (no files written)
- default real-run mode uses `overwrite=False`
- existing files are never deleted
- existing files are not overwritten unless you explicitly set `overwrite=True`

### Two combine options: standard vs concat-with-batch

1. `anndata_builder.combined_anndata(join="outer")`
- Uses `anndata.concat` on a list of component objects.
- Produces combined AnnData without adding a new `batch` column.
- Supports join behavior via `join` (`"outer"` or `"inner"`).

2. `anndata_builder.combined_anndata_concat(join="outer", label="batch")`
- Uses `anndata.concat` on a dict keyed by `run_sample_id`.
- Adds a `batch` column in `.obs` where values are the `run_sample_id` keys.
- Also keeps explicit `run_sample_id` in `.obs`.
- Supports join behavior via `join` (`"outer"` or `"inner"`).

### Mental model

- `anndata_builder = ...` is configuration.
- `sample_table()` is metadata I/O and label harmonization.
- `combined_anndata()` or `combined_anndata_concat()` is matrix I/O + full object construction.
- raw save utility writes per-sample AnnData files only after a dry-run review.

In [3]:
# Tutorial cell: instantiate Raw10xAnnDataBuilder with explicit, traceable intent.
#

# -----------------------------------------------------------------------------
# 1) Find project root (pure path discovery; no data loading yet)
# -----------------------------------------------------------------------------
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "metadata" / "div30_div90_sample_id_to_biolabel_map.tsv").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing metadata/div30_div90_sample_id_to_biolabel_map.tsv")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())

#print the project root to confirm it was found correctly
print("Project root found at:", PROJECT_ROOT)

# -----------------------------------------------------------------------------
# 2) Instantiate the configuration/helper object (no matrix I/O yet)
# -----------------------------------------------------------------------------
anndata_builder = Raw10xAnnDataBuilder(
    project_root=PROJECT_ROOT,
    target_divs=("DIV30",),
    target_run_sample_ids=(
        "9853-MW-1",
        "9853-MW-2",
        "9853-MW-3",
        "9853-MW-4",
        "9853-MW-5",
        "9853-MW-6",
    ),
)

# -----------------------------------------------------------------------------
# 3) Introspect what was created right now
# -----------------------------------------------------------------------------
print("Type(anndata_builder):", type(anndata_builder))
print("Builder configuration currently stored:")
print("  project_root:", anndata_builder.project_root)
print("  target_divs:", anndata_builder.target_divs)
print("  target_run_sample_ids:", anndata_builder.target_run_sample_ids)
print("  sample_map_name (default):", anndata_builder.sample_map_name)
print("  strict_missing_matrix_dirs (default):", anndata_builder.strict_missing_matrix_dirs)
print("  sample_map_tsv path:", anndata_builder.sample_map_tsv)

# -----------------------------------------------------------------------------
# 4) First operation that DOES perform I/O: sample_table()
# -----------------------------------------------------------------------------
sample_metadata_df = anndata_builder.sample_table()
print(f"Selected {len(sample_metadata_df)} samples from: {anndata_builder.sample_map_tsv}")
display(sample_metadata_df[["DIV", "run_sample_id", "cell_line", "matrix_dir"]])

Project root found at: /home/elcrespo/Desktop/githubprojects/mge_organoid_pipeline
Type(anndata_builder): <class '__main__.Raw10xAnnDataBuilder'>
Builder configuration currently stored:
  project_root: /home/elcrespo/Desktop/githubprojects/mge_organoid_pipeline
  target_divs: ('DIV30',)
  target_run_sample_ids: ('9853-MW-1', '9853-MW-2', '9853-MW-3', '9853-MW-4', '9853-MW-5', '9853-MW-6')
  sample_map_name (default): metadata/div30_div90_sample_id_to_biolabel_map.tsv
  strict_missing_matrix_dirs (default): True
  sample_map_tsv path: /home/elcrespo/Desktop/githubprojects/mge_organoid_pipeline/metadata/div30_div90_sample_id_to_biolabel_map.tsv
Selected 6 samples from: /home/elcrespo/Desktop/githubprojects/mge_organoid_pipeline/metadata/div30_div90_sample_id_to_biolabel_map.tsv


,DIV,run_sample_id,cell_line,matrix_dir
0,DIV30,9853-MW-1,H9,/nfs/turbo/umms-parent/Manny_test/9583-MW-rean...
1,DIV30,9853-MW-2,H9,/nfs/turbo/umms-parent/Manny_test/9583-MW-rean...
2,DIV30,9853-MW-3,79B,/nfs/turbo/umms-parent/Manny_test/9583-MW-rean...
3,DIV30,9853-MW-4,79B,/nfs/turbo/umms-parent/Manny_test/9583-MW-rean...
4,DIV30,9853-MW-5,2E,/nfs/turbo/umms-parent/Manny_test/9583-MW-rean...
5,DIV30,9853-MW-6,2E,/nfs/turbo/umms-parent/Manny_test/9583-MW-rean...


In [4]:
# Standalone utility: save one raw AnnData file per sample to $PROJECT_ROOT/raw_adata.
# Filename prefix is derived from the Cell Ranger matrix path.
def save_raw_adatas_from_builder(
    anndata_builder: Raw10xAnnDataBuilder,
    output_dir: Path = None,
    compression: str = "gzip",
    dry_run: bool = True,
    overwrite: bool = False,
) -> pd.DataFrame:
    sample_metadata_df, sample_adatas_by_run = anndata_builder._load_component_adatas()

    # Default write location is driven by the exported PROJECT_ROOT environment variable.
    if output_dir is None:
        project_root_env = os.environ.get("PROJECT_ROOT")
        if not project_root_env:
            raise EnvironmentError(
                "PROJECT_ROOT is not set. Export it first, e.g. "
                "export PROJECT_ROOT=/nfs/turbo/umms-parent/mgeo_neuron_scrnaseq_projectfolder"
            )
        output_dir = Path(project_root_env) / "raw_adata"

    output_dir = Path(output_dir)
    if not dry_run:
        output_dir.mkdir(parents=True, exist_ok=True)

    sample_meta_by_run = sample_metadata_df.copy()
    sample_meta_by_run["run_sample_id_str"] = sample_meta_by_run["run_sample_id"].astype(str)
    sample_meta_by_run = sample_meta_by_run.set_index("run_sample_id_str")

    write_records = []
    planned_paths = set()

    for run_sample_id, one_sample_adata in sample_adatas_by_run.items():
        matrix_dir = Path(sample_meta_by_run.loc[run_sample_id, "matrix_dir"])

        # Typical pattern: <prefix>/count/sample_filtered_feature_bc_matrix
        if "count" in matrix_dir.parts:
            count_idx = matrix_dir.parts.index("count")
            path_prefix = matrix_dir.parts[count_idx - 1] if count_idx > 0 else matrix_dir.parent.name
        else:
            path_prefix = matrix_dir.parent.name

        safe_prefix = str(path_prefix).replace(" ", "_")
        candidate_path = output_dir / f"{safe_prefix}.h5ad"

        # Never overwrite existing files unless explicitly requested.
        out_path = candidate_path
        if (not overwrite) and (out_path.exists() or str(out_path) in planned_paths):
            # Keep the matrix-path prefix but make filename unique for this sample.
            out_path = output_dir / f"{safe_prefix}__{run_sample_id}.h5ad"

        planned_paths.add(str(out_path))

        action = "dry_run"
        wrote_file = False

        if not dry_run:
            if out_path.exists() and not overwrite:
                action = "skipped_existing"
            else:
                # Use write_h5ad explicitly, as requested.
                one_sample_adata.write_h5ad(out_path, compression=compression)
                action = "written"
                wrote_file = True

        write_records.append(
            {
                "dry_run": bool(dry_run),
                "action": action,
                "wrote_file": wrote_file,
                "overwrite_enabled": bool(overwrite),
                "run_sample_id": run_sample_id,
                "cell_line": str(sample_meta_by_run.loc[run_sample_id, "cell_line"]),
                "matrix_dir": str(matrix_dir),
                "file_prefix": safe_prefix,
                "h5ad_path": str(out_path),
                "exists_after_run": out_path.exists(),
                "n_cells": int(one_sample_adata.n_obs),
                "n_genes": int(one_sample_adata.n_vars),
            }
        )

    return pd.DataFrame(write_records).sort_values(["file_prefix", "run_sample_id"]).reset_index(drop=True)

In [5]:
real_run_manifest_df = save_raw_adatas_from_builder(anndata_builder, dry_run=False, overwrite=False)
print("Written files:", (real_run_manifest_df["action"] == "written").sum())
print("Skipped existing files:", (real_run_manifest_df["action"] == "skipped_existing").sum())
print("Output directory:", Path(real_run_manifest_df["h5ad_path"].iloc[0]).parent)
display(real_run_manifest_df[["action", "run_sample_id", "cell_line", "file_prefix", "h5ad_path", "exists_after_run", "n_cells", "n_genes"]])

Written files: 0
Skipped existing files: 6
Output directory: /nfs/turbo/umms-parent/mgeo_neuron_scrnaseq_projectfolder/raw_adata


,action,run_sample_id,cell_line,file_prefix,h5ad_path,exists_after_run,n_cells,n_genes
0,skipped_existing,9853-MW-1,H9,9853-MW-1,/nfs/turbo/umms-parent/mgeo_neuron_scrnaseq_pr...,True,18047,18082
1,skipped_existing,9853-MW-2,H9,9853-MW-2,/nfs/turbo/umms-parent/mgeo_neuron_scrnaseq_pr...,True,6060,18082
2,skipped_existing,9853-MW-3,79B,9853-MW-3,/nfs/turbo/umms-parent/mgeo_neuron_scrnaseq_pr...,True,17928,18082
3,skipped_existing,9853-MW-4,79B,9853-MW-4,/nfs/turbo/umms-parent/mgeo_neuron_scrnaseq_pr...,True,13047,18082
4,skipped_existing,9853-MW-5,2E,9853-MW-5,/nfs/turbo/umms-parent/mgeo_neuron_scrnaseq_pr...,True,26408,18082
5,skipped_existing,9853-MW-6,2E,9853-MW-6,/nfs/turbo/umms-parent/mgeo_neuron_scrnaseq_pr...,True,25530,18082


In [ ]:
# Build an ordered list of per-sample AnnData objects without concatenating them.
sample_metadata_df, sample_adatas_by_run = anndata_builder._load_component_adatas()

adata_names = sample_metadata_df["run_sample_id"].astype(str).tolist()
adata_list = [sample_adatas_by_run[run_sample_id] for run_sample_id in adata_names]

print("Number of AnnData objects:", len(adata_list))
print("Sample IDs:", adata_names)
print("All objects are AnnData:", all(isinstance(one_sample_adata, ad.AnnData) for one_sample_adata in adata_list))

for run_sample_id, one_sample_adata in zip(adata_names, adata_list):
    unique_run_ids = one_sample_adata.obs["run_sample_id"].nunique()
    print(
        run_sample_id,
        one_sample_adata.shape,
        "unique_cell_ids=",
        one_sample_adata.obs_names.is_unique,
        "obs_run_sample_id_count=",
        unique_run_ids,
    )

In [ ]:
# Build the standard combined AnnData object for downstream Scanpy work.
# Option A (default path): no additional batch column created.
#combined_adata = anndata_builder.combined_anndata(join="outer")

# Option B (alternate path): adds .obs['batch'] using run_sample_id keys.
# this is a combined version....so there is one matrix this is NOT a list of adatas to be combined
combined_adata = anndata_builder.combined_anndata_concat(join="outer", label="batch")

print("Combined AnnData shape:", combined_adata.shape)
print("Loaded run_sample_id count:", combined_adata.obs["run_sample_id"].nunique())
print("Unique cell IDs:", combined_adata.obs_names.is_unique)
print("Has batch column:", "batch" in combined_adata.obs.columns)

Combined AnnData shape: (107020, 18082)
Loaded run_sample_id count: 6
Unique cell IDs: True
Has batch column: True
